In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:


from trainer.transformer_utilts import *
sys.path.append("..")
import torch.optim as optim
from collections import Counter
from tqdm import tqdm
import pandas as pd
import time 
import torch
from dotenv import load_dotenv
from model.MF import get_model,get_tokenizer
from helper.dataloader import *
import wandb
from tqdm import tqdm
from peft import LoraConfig, TaskType, PeftModel,get_peft_model
from transformers import T5Tokenizer ,AutoTokenizer
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from transformers import AutoConfig
from transformers import AutoModel
from model.eval_model import *
# torch.backends.cuda.enable_mem_efficient_sdp(False)
# torch.backends.cuda.enable_flash_sdp(False)
import os
import os
import sys 
import time
import pandas as pd 
import numpy as np
import torch
import openai
from tqdm import tqdm
from dotenv import load_dotenv
from trainer.transformer_utilts import *
from helper.dataloader import * 
import pickle
from peft import get_peft_model

import argparse
from tqdm import tqdm
from peft import LoraConfig, TaskType
import logging
from transformers import AutoTokenizer
from collections import Counter

rank =0
world_size = 1 
args = parse_args(notebook=True)
args.embedding_module="OTRecVAE"


ModuleNotFoundError: No module named 'torch'

# Change this path to where you stored the models

In [ ]:
args.scratch = 'home/mila/a/adls/saved_model/ml-1m/'

In [ ]:
args.data_name ='ml-1m'

labels_map = {'OTRecVAE' : 'TEARS RecVAE','T5Vae':'TEARS Multi-VAE','VAE':'MultiVAE','RecVAE':'RecVAE','VariationalT5':'TEARS-Base',
                'DAE':'Multi-DAE','EASE':'EASE', 'MacridTEARS':'TEARS MacridVAE','MacridVAE':'MacridVAE',
                'GenreTEARS':'GenreTEARS','RecVAEGenreVAE':'GenreTEARS-RecVAE'}
if args.data_name == 'ml-1m':


    paths = {
            'OTRecVAE': 'ot_train_vae_ml-1m_embedding_module_OTRecVAE_2024-09-24_13-37-29_2022.csv',
    }

item_genre_dict = map_id_to_genre(args.data_name)

item_title_dict = map_id_to_title(args.data_name)
alpha = .5
tokenizer = get_tokenizer(args)



In [ ]:

args.bs = 250 if args.data_name !='goodbooks' else 1000
args.llm_backbone = 'gpt-4-1106-preview'
prompts,rec_dataloader,augmented_dataloader,num_movies,val_dataloader,test_dataloader,val_data_tr,test_data_tr= load_data(args,tokenizer,rank,world_size)

In [ ]:
alphas = [x/10 for x in range(0,11)]
loss = get_loss(args.loss)
out_metrics = defaultdict(dict)
ups = defaultdict(list)
downs = defaultdict(list)
ndcgs20 = defaultdict(list)
best_alpha = {}
qualitative_metrics = {}

In [ ]:



args.concat = False
args.kfac = 2
# args.tau = 0
for module,path in paths.items():
    if path == '': continue
    p = f'{args.scratch}/saved_model/{args.data_name}/' + path+'.pt'

    
    args.embedding_module = module
    args.lora_r = 64
    args.bs =64
    model,lora_config = get_model(args, tokenizer, num_movies, 0, world_size)
    if module in ['T5Vae','VariationalT5']:
        model = get_peft_model(model, lora_config)           
        print('gotpeft')

    model.to(rank)
    state_dict = torch.load(p, map_location=torch.device('cuda'))
    model.load_state_dict(state_dict)


In [ ]:


summary = \
    """
    Summary: The user enjoys a variety of genres, with a particular affinity \
    for kids movies and animated films. They appreciate films that blend humor with other elements,\
    such as animated children's movies that incorporate comedy, or action and sci-fi films that manage to weave in comedic relief.\
    The user also shows a strong preference for war dramas and films that combine action with a deeper dramatic storyline,\
    indicating a taste for intense emotional experiences and complex character development.\n\n\
    Plot points that resonate with the user include intricate storylines with a blend of\
    suspense and humor, as well as narratives that explore the human condition against the backdrop of larger societal issues or\
    historical events. The user seems to enjoy clever dialogue and narratives that challenge expectations,\
    possibly with twists or unconventional story arcs.\n\nOn the other hand, the user does not enjoy certain crime thrillers and dramas\
    that perhaps focus too heavily on the darker aspects of the genre without the balance of humor or other engaging elements.\They also seem less interested\
    in films that may have a slower pace or lack the dynamic interplay of genres that they typically favor.\
    \n\nOther users may appreciate the tension and psychological\
    intrigue of pure thrillers or the raw, realistic portrayal of events in certain dramas,\
    even if these elements are less appealing to this particular user.
    """


data_tensor = torch.zeros((num_movies)).to(rank) - 1e20
ranked_logits,logits = model.generate_recommendations(summary,tokenizer = tokenizer, data_tensor = data_tensor,topk = 20,alpha = 0)


indices_to_movies(ranked_logits)

